<a href="https://colab.research.google.com/github/IamRabin/ExCIR/blob/main/ExCIRBlockCIR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# excir_bounds_demo.py — fixed LB/UB plotting, robust MMD^2, drift vs q, timing, shading, annotations
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter

# ---------- metrics ----------
def median_pairwise_distance_pooled(A, B, max_pairs=20000):
    """Median pairwise distance on pooled data (for RBF bandwidth)."""
    P = np.vstack([A, B])
    n = P.shape[0]
    if n * (n - 1) // 2 > max_pairs:
        idx = np.random.choice(n, size=int(np.sqrt(2 * max_pairs)) + 1, replace=False)
        P = P[idx]
    d2 = np.sum((P[:, None, :] - P[None, :, :]) ** 2, axis=-1)
    iu = np.triu_indices_from(d2, k=1)
    med = np.median(np.sqrt(d2[iu]))
    return max(med, 1e-6)

def pairwise_d2(X, Y):
    X2 = np.sum(X**2, axis=1, keepdims=True)
    Y2 = np.sum(Y**2, axis=1, keepdims=True).T
    return X2 + Y2 - 2.0 * X @ Y.T

def mmd2_gaussian_multiscale(Y, Yp, scales=(0.5, 1.0, 2.0, 4.0)):
    """Unbiased MMD^2 with Gaussian kernel, pooled bandwidth + multiscale."""
    base_sigma = median_pairwise_distance_pooled(Y, Yp)
    n, m = Y.shape[0], Yp.shape[0]
    d2_xx = pairwise_d2(Y, Y);   np.fill_diagonal(d2_xx, 0.0)
    d2_yy = pairwise_d2(Yp, Yp); np.fill_diagonal(d2_yy, 0.0)
    d2_xy = pairwise_d2(Y, Yp)
    vals = []
    for s in scales:
        sigma = base_sigma * s
        Kxx = np.exp(-d2_xx / (2 * sigma**2))
        Kyy = np.exp(-d2_yy / (2 * sigma**2))
        Kxy = np.exp(-d2_xy / (2 * sigma**2))
        term_xx = Kxx.sum() / (n * (n - 1)) if n > 1 else 0.0
        term_yy = Kyy.sum() / (m * (m - 1)) if m > 1 else 0.0
        term_xy = 2.0 * Kxy.mean()
        vals.append(max(term_xx + term_yy - term_xy, 0.0))
    return float(np.mean(vals))

def projection_alignment(Y, Yp):
    """D_proj = ||Y - (Yp A + 1 b^T)||_F / ||Y||_F with LS A,b."""
    n, q = Y.shape
    Z = np.hstack([Yp, np.ones((n, 1))])
    theta, *_ = np.linalg.lstsq(Z, Y, rcond=None)
    A, b = theta[:-1, :], theta[-1, :]
    resid = Y - (Yp @ A + np.ones((n, 1)) @ b[None, :])
    return float(np.linalg.norm(resid, "fro") / (np.linalg.norm(Y, "fro") + 1e-12))

def kde_1d(x, grid, h):
    diffs = (grid[:, None] - x[None, :]) / h
    ker = np.exp(-0.5 * diffs**2) / (np.sqrt(2 * np.pi) * h)
    dens = ker.mean(axis=1)
    return np.maximum(dens, 1e-12)

def axiswise_kl(Y, Yp, grid_pts=160):
    """Max over axes of 1D KDE KL(p||q) on standardized coordinates."""
    n, q = Y.shape
    vals = []
    for j in range(q):
        x = (Y[:, j] - Y[:, j].mean()) / (Y[:, j].std() + 1e-8)
        y = (Yp[:, j] - Yp[:, j].mean()) / (Yp[:, j].std() + 1e-8)
        lo = min(x.min(), y.min()) - 3.0
        hi = max(x.max(), y.max()) + 3.0
        grid = np.linspace(lo, hi, grid_pts)
        hx = max(1.06 * np.std(x) * (len(x) ** (-1 / 5)), 1e-2)
        hy = max(1.06 * np.std(y) * (len(y) ** (-1 / 5)), 1e-2)
        px = kde_1d(x, grid, hx)
        py = kde_1d(y, grid, hy)
        dx = (hi - lo) / (grid_pts - 1)
        vals.append(float(np.sum(px * (np.log(px) - np.log(py))) * dx))
    return float(np.max(vals))

# ---------- synthetic outputs ----------
def synth_full_outputs(n_eval, q, signal_dim=4, noise=0.2):
    Z = np.random.randn(n_eval, signal_dim)
    W = np.random.randn(signal_dim, q)
    return Z @ W + noise * np.random.randn(n_eval, q)

def random_orthogonal(q):
    Q, _ = np.linalg.qr(np.random.randn(q, q))
    return Q

def synth_lightweight_outputs(Y_full, nprime, sep=0.9):
    """
    Distortion decreases with n' and grows with sqrt(q): rotation + scale + noise.
    """
    n_eval, q = Y_full.shape
    eps = (sep * np.sqrt(q)) / np.sqrt(max(nprime, 1) / 500.0 + 1.0)
    R = random_orthogonal(q)
    A = np.eye(q) + eps * R
    b = (eps * 0.25) * np.random.randn(q)
    noise = (eps * 0.35) * (Y_full.std() + 1e-8) * np.random.randn(n_eval, q)
    return Y_full @ A + b + noise

# ---------- plotting ----------
def plot_mmd2_vs_nprime(df, out_path):
    plt.figure(figsize=(8, 5.0))
    all_vals = []
    for q in sorted(df["q"].unique()):
        sub = df[df["q"] == q].sort_values("nprime")
        y = sub["MMD2"].values
        all_vals.append(y)
        plt.plot(sub["nprime"].values, y, marker="o", linewidth=2, label=f"q={q}")
    all_vals = np.concatenate(all_vals) if len(all_vals) else np.array([1e-9])
    plt.xlabel("$n'$ (lightweight sample size)")
    plt.ylabel("Empirical MMD$^2$")
    plt.title("MMD$^2$ vs. $n'$ across output dimensions")
    plt.grid(True, linestyle="--", linewidth=0.6, alpha=0.6)
    plt.yscale("log")
    ymin = max(np.nanmin(all_vals) * 0.7, 1e-8)
    ymax = max(np.nanmax(all_vals) * 1.3, ymin * 10)
    plt.ylim(ymin, ymax)
    plt.legend(frameon=False)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()

# ---------- core experiment ----------
def run_experiment(
    save_dir="excir_bounds_demo_out",
    n_eval=800,
    q_list=(1, 2, 4, 8, 16, 32),
    nprime_grid=np.unique(np.logspace(np.log10(100), np.log10(12000), 50).astype(int)),
    eps_acc=0.2, # Changed to a less-strict value
    L_l=1.0, C_mmd=0.3, C_kl=0.3, # Changed to less-strict values
    time_budget_sec=0.2,
    min_pass=3,
    seed=33,
):
    """
    This function runs the core experiment to determine the statistical and operational
    bounds for the sample size n' based on various metrics and runtime constraints.
    It generates synthetic data, performs metric calculations, and then plots the
    lower and upper bounds to find a feasible operating window.
    """
    np.random.seed(seed)
    out = Path(save_dir); out.mkdir(parents=True, exist_ok=True)

    # per-metric tolerances
    eps_proj = eps_acc / (3.0 * L_l)
    eps_mmd  = (eps_acc / (3.0 * C_mmd))**2
    eps_kl   = (eps_acc / (3.0 * C_kl))**2

    rows = []
    for q in q_list:
        Y = synth_full_outputs(n_eval, q)
        for nprime in nprime_grid:
            t0 = time.time()
            Yp = synth_lightweight_outputs(Y, nprime, sep=0.9)
            D_proj = projection_alignment(Y, Yp)
            MMD2   = mmd2_gaussian_multiscale(Y, Yp)
            KLax   = axiswise_kl(Y, Yp)
            cost_reps = int(0.4 * q)
            for _ in range(cost_reps):
                _ = pairwise_d2(Y[:200], Yp[:200])
            elapsed = time.time() - t0
            flags = [D_proj <= eps_proj, MMD2 <= eps_mmd, KLax <= eps_kl]
            pass_count = sum(int(b) for b in flags)
            rows.append({
                "q": q, "nprime": nprime,
                "D_proj": D_proj, "MMD2": MMD2, "KL_axiswise": KLax,
                "pass_proj": flags[0], "pass_mmd": flags[1], "pass_kl": flags[2],
                "pass_count": pass_count,
                "all_pass": (pass_count == 3),
                "maj_pass": (pass_count >= 2),
                "runtime_sec": elapsed
            })

    df = pd.DataFrame(rows).sort_values(["q", "nprime"])
    df.to_csv(out / "metrics_results.csv", index=False)

    # --- LB/UB computation ---
    lb_any = df[df["pass_count"] >= min_pass].groupby("q")["nprime"].min()
    ub_emp = df[df["runtime_sec"] <= time_budget_sec].groupby("q")["nprime"].max()

    qs = sorted(df["q"].unique())
    lb_vals = [lb_any.get(q, np.nan) for q in qs]
    ub_vals = [ub_emp.get(q, np.nan) for q in qs]

    # --- two-line plot ---
    plt.figure(figsize=(8.2, 5.6))

    # Plot the lower bound line with a label
    plt.plot(qs, lb_vals, marker="o", linewidth=2.5, markersize=7,
             label="Statistical lower bound (min $n'$)")

    # Plot the upper bound line with a label
    plt.plot(qs, ub_vals, marker="s", linewidth=2.5, markersize=7,
             label=f"Operational upper bound (runtime ≤ {int(time_budget_sec)}s)")

    # Shade the feasible window where LB is less than or equal to UB
    x = np.asarray(qs, float)
    y1 = np.asarray(lb_vals, float)
    y2 = np.asarray(ub_vals, float)

    plt.fill_between(x, y1, y2, where=(y2 >= y1), color="tab:grey", alpha=0.15,
                     step="mid", label="Feasible $n'$ window")

    # Annotate values on the plot
    if np.isfinite(np.nanmax(ub_vals)) and np.isfinite(np.nanmin(lb_vals)):
        bump = 0.02 * (np.nanmax(ub_vals) - np.nanmin(lb_vals) + 1.0)
    else:
        bump = 50.0

    for x0, y0 in zip(qs, lb_vals):
        if np.isfinite(y0):
            plt.text(x0, y0 + bump, f"{int(y0):,}", ha="center", va="bottom", fontsize=9)
    for x0, y0 in zip(qs, ub_vals):
        if np.isfinite(y0):
            plt.text(x0, y0 + bump, f"{int(y0):,}", ha="center", va="bottom",
                     fontsize=9, color="tab:orange")

    plt.xlabel("Output dimension q", fontsize=12)
    plt.ylabel("n' (samples)", fontsize=12)
    plt.title("Lower vs. Upper bounds on n' across output dimensions", fontsize=14)
    plt.gca().yaxis.set_major_formatter(StrMethodFormatter('{x:,.0f}'))
    plt.grid(True, linestyle="--", linewidth=0.6, alpha=0.6)
    plt.legend(frameon=False, fontsize=10)
    plt.tight_layout()
    fig_path = out / "lb_ub_two_lines.png"
    plt.savefig(fig_path, dpi=150)
    plt.close()

    # MMD^2 curves
    plot_mmd2_vs_nprime(df, out / "mmd2_vs_nprime.png")

    # table
    pd.DataFrame({"q": qs, "LB_nprime": lb_vals, "UB_nprime": ub_vals}).to_csv(
        out / "lb_ub_table.csv", index=False
    )

    print("Saved:", fig_path)
    print("Saved:", out / "mmd2_vs_nprime.png")
    print("Saved:", out / "metrics_results.csv")
    print("Saved:", out / "lb_ub_table.csv")
    return df

# ---------- run ----------
if __name__ == "__main__":
    run_experiment(
        save_dir="excir_bounds_demo_out",
        n_eval=800,
        q_list=(1, 2, 4, 8, 16, 32),
        nprime_grid=np.unique(np.logspace(np.log10(100), np.log10(12000), 50).astype(int)),
        eps_acc=0.2, # Changed to a less-strict value
        L_l=1.0,
        C_mmd=0.3, # Changed to a less-strict value
        C_kl=0.3,  # Changed to a less-strict value
        time_budget_sec=0.2,
        min_pass=3,
        seed=33,
    )


In [ ]:
!pip install shap

In [ ]:
# ========================== CONFIG ==========================
TASK        = "classification"     # keep "classification" for this demo
N_SAMPLES   = 6000                 # synthetic rows to generate
TEST_SIZE   = 0.20                 # test split
VAL_SIZE    = 0.20                 # share of (train) held out for validation
LIGHT_FRAC  = 0.35                 # fraction for lightweight subset (0.2–0.5 typical)
RANDOM_SEED = 42                   # reproducibility
TOP_K_LIST  = [6, 8]               # for sufficiency table
TOPN_PLOTS  = 20                   # how many features to show in the CIR plot
# ============================================================

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.utils import check_random_state
from scipy.special import expit as sigmoid
import shap

rng = check_random_state(RANDOM_SEED)

# ---------- 1) Generate synthetic vehicular dataset ----------
feat_names = [
    "speed_kph","rpm","throttle","brake","steering_deg","gear",
    "accel_long","accel_lat","yaw_rate","road_grade",
    "ambient_temp","tire_fl","tire_fr","tire_rl","tire_rr",
    "engine_load","maf","intake_air_temp","battery_v","fuel_rate"
]

n = N_SAMPLES
speed = np.clip(rng.normal(80, 15, n), 0, None)
throttle = np.clip(rng.beta(2, 2, n), 0, 1)
brake = np.clip(1 - throttle + rng.normal(0, 0.15, n), 0, 1)
steering = rng.normal(0, 10, n)
gear = np.clip((speed // 20) + rng.normal(0.0, 0.5, n), 1, 7)
accel_long = rng.normal(0.05*throttle*speed - 0.08*brake*speed, 0.5, n)
accel_lat = rng.normal(np.abs(steering)/18 * (speed/80), 0.2, n)
yaw_rate = rng.normal(steering/30 * (speed/60), 0.2, n)
road_grade = rng.normal(0, 2, n)
ambient_temp = rng.normal(20, 8, n)
tire_base = rng.normal(34, 1.0, (n,4))
low_mask = rng.uniform(0,1,n) < 0.15
tire_drop = rng.normal(4, 1.0, (n,4)) * low_mask[:,None]
tires = tire_base - tire_drop
engine_load = np.clip(30 + 50*throttle + 5*road_grade + rng.normal(0, 5, n), 0, 100)
maf = np.clip(5 + 0.06*speed + 0.5*engine_load/100 + rng.normal(0,0.7,n), 0, None)
intake_air_temp = np.clip(ambient_temp + rng.normal(10, 2, n), -10, 80)
battery_v = np.clip(rng.normal(13.8, 0.3, n) - 0.2*brake + 0.05*(engine_load/100), 11.5, 15)
fuel_rate = np.clip(0.5 + 0.02*speed + 0.6*throttle + 0.1*(engine_load/100) + rng.normal(0,0.2,n), 0, None)
rpm = np.clip(800 + 35*speed + 1200*throttle + rng.normal(0, 300, n), 700, 7000)

X_df = pd.DataFrame({
    "speed_kph": speed,
    "rpm": rpm,
    "throttle": throttle,
    "brake": brake,
    "steering_deg": steering,
    "gear": gear,
    "accel_long": accel_long,
    "accel_lat": accel_lat,
    "yaw_rate": yaw_rate,
    "road_grade": road_grade,
    "ambient_temp": ambient_temp,
    "tire_fl": tires[:,0],
    "tire_fr": tires[:,1],
    "tire_rl": tires[:,2],
    "tire_rr": tires[:,3],
    "engine_load": engine_load,
    "maf": maf,
    "intake_air_temp": intake_air_temp,
    "battery_v": battery_v,
    "fuel_rate": fuel_rate,
})

# --- FIXED LINE (use Pandas row-wise min + Series.clip) ---
low_tire = (32 - X_df[["tire_fl","tire_fr","tire_rl","tire_rr"]].min(axis=1)).clip(lower=0)

risk_logit = (
    1.2*(X_df["speed_kph"]-110)/20
    + 1.1*X_df["brake"]
    + 0.9*np.abs(X_df["steering_deg"])/15
    + 0.7*np.abs(X_df["yaw_rate"])
    + 0.8*(low_tire)
    + 0.7*(X_df["engine_load"]/100)
    + 0.3*(X_df["road_grade"]/5)
    - 0.2*(X_df["battery_v"]-13.5)
)
p = sigmoid(risk_logit + rng.normal(0, 0.4, n))
y = (rng.uniform(0,1,n) < p).astype(int)

# ---------- 2) Split & scale ----------
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X_df, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=VAL_SIZE, stratify=y_train_full, random_state=RANDOM_SEED
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

# ---------- 3) Train original model ----------
Model = GradientBoostingClassifier
orig_model = Model(random_state=RANDOM_SEED).fit(
    scaler.fit_transform(pd.concat([X_train, X_val], axis=0)),
    np.concatenate([y_train, y_val], axis=0)
)

def model_outputs(model, Xs):
    return model.predict_proba(Xs)[:, -1]

yhat_val_orig  = model_outputs(orig_model, scaler.transform(X_val))
yhat_test_orig = model_outputs(orig_model, scaler.transform(X_test))

# ---------- 4) CIR computation ----------
def compute_cir(Xs, yhat, names):
    n = Xs.shape[0]
    y_bar = yhat.mean()
    vals = []
    for i in range(Xs.shape[1]):
        f = Xs[:, i]
        f_bar = f.mean()
        m = 0.5*(f_bar + y_bar)
        num = n*((f_bar - m)**2 + (y_bar - m)**2)
        den = np.sum((f - m)**2) + np.sum((yhat - m)**2)
        eta = float(num/den) if den > 0 else 0.0
        vals.append(eta)
    return pd.Series(vals, index=names).sort_values(ascending=False)

cir_orig_val = compute_cir(scaler.transform(X_val), yhat_val_orig, X_df.columns.tolist())

# ---------- 5) Lightweight environment ----------
full_Xs = scaler.fit_transform(pd.concat([X_train, X_val], axis=0))
full_y  = np.concatenate([y_train, y_val], axis=0)
full_df = pd.DataFrame(full_Xs, columns=X_df.columns).assign(target=full_y)

light_df = full_df.groupby("target", group_keys=False).apply(
    lambda g: g.sample(max(1, int(len(g)*LIGHT_FRAC)), random_state=RANDOM_SEED)
).reset_index(drop=True)

X_light_s = light_df[X_df.columns].values
y_light   = light_df["target"].values

light_model = Model(random_state=RANDOM_SEED).fit(X_light_s, y_light)
yhat_val_light = model_outputs(light_model, scaler.transform(X_val))
cir_light_val  = compute_cir(scaler.transform(X_val), yhat_val_light, X_df.columns.tolist())

# ---------- 6) Figure: CIR Original vs Lightweight ----------
order = cir_orig_val.index[:TOPN_PLOTS]
vals_o = cir_orig_val.loc[order].values
vals_l = cir_light_val.loc[order].values

plt.figure(figsize=(10, 6))
ypos = np.arange(len(order))
bar_h = 0.35
plt.barh(ypos+bar_h/2, vals_l, height=bar_h, label="Lightweight Space")
plt.barh(ypos-bar_h/2, vals_o, height=bar_h, label="Original Space")
plt.yticks(ypos, order)
plt.xlabel("Feature Score (CIR)")
plt.title("Feature Importance Comparison (Lightweight vs Original)")
plt.legend()
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# ---------- 7) SHAP bar chart (Original model) ----------
explainer = shap.TreeExplainer(orig_model)
val_sample = min(800, X_val_s.shape[0])
idx = rng.choice(X_val_s.shape[0], size=val_sample, replace=False)
shap_vals = explainer.shap_values(X_val_s[idx])
if isinstance(shap_vals, list):
    shap_mat = shap_vals[-1]
else:
    shap_mat = shap_vals
mean_abs_shap = np.abs(shap_mat).mean(axis=0)
shap_series = pd.Series(mean_abs_shap, index=X_df.columns).sort_values(ascending=False)

topN_shap = 10
shap_top = shap_series.head(topN_shap)
plt.figure(figsize=(7, 4))
plt.barh(np.arange(len(shap_top))[::-1], shap_top.values[::-1])
plt.yticks(np.arange(len(shap_top))[::-1], shap_top.index[::-1])
plt.xlabel("Mean |SHAP| value")
plt.title("Top features by SHAP value (Original model)")
plt.tight_layout()
plt.show()

# ---------- 8) Top-k sufficiency table ----------
def train_eval_on_features(cols):
    idxs = [list(X_df.columns).index(c) for c in cols]
    X_full_sub = full_Xs[:, idxs]
    X_test_sub = scaler.transform(X_test)[:, idxs]
    m = Model(random_state=RANDOM_SEED).fit(X_full_sub, full_y)
    ypred = m.predict(X_test_sub)
    return accuracy_score(y_test, ypred) * 100.0

rows = []
for k in TOP_K_LIST:
    excir_feats = cir_orig_val.head(k).index.tolist()
    shap_feats  = shap_series.head(k).index.tolist()
    acc_excir = train_eval_on_features(excir_feats)
    acc_shap  = train_eval_on_features(shap_feats)
    rows.append(["SHAP-Ranked Features", k, acc_shap])
    rows.append(["ExCIR-Ranked Features", k, acc_excir])

acc_table = pd.DataFrame(rows, columns=["Method", "No. of Features", "Accuracy (%)"])
print("\nTop-k sufficiency results:")
print(acc_table.to_string(index=False))
